# Common and specific TF interface P/LP enrichment

Two separately constructed statistical cohorts are analyzed: specific (123 TFs with P/LP variants) and common (42 TFs with P/LP variants). These are not independent cohorts: the historical inputs overlap by 40 TFs and 837 P/LP variants. The upstream networks span 14 cell types: common edges occur in all 14; specific edges in exactly one. They are not the four-lineage top-5% classes in notebook 16.

| Cohort | Network TFs | P/LP TFs | Interface P/LP variants / all P/LP variants | Interface / full-length residues in the same P/LP TFs |
| --- | ---: | ---: | ---: | ---: |
| Specific | 392 | 123 | 371 / 2,073 | 6,665 / 79,071 |
| Common | 116 | 42 | 120 / 841 | 1,604 / 25,143 |

These are expected values for the archived inputs, not outputs generated by opening this notebook. Variant counts differ from unique residue counts. Do not substitute the historical 5.83% and 7.88% all-variant-site statistics.

Interfaces are cross-chain all-atom contacts at <=5 angstroms, intersected across all five structural samples per pair and then unioned across partners. No pLDDT or SASA filter is applied. Clinical inputs are preverified simple missense variants with standard PLP/BLB/VUS classes, all origins, and no review-status filter. Upstream reference alignment required >=95% identity and uniquely mapped positions. Cached gnomAD r4 exome controls use PASS, AC>0 and AN>=80% of the gene maximum, excluding known ClinVar P/LP genomic alleles upstream. Those annotation and sequence-mapping steps must already be complete.

The first analysis uses a within-TF residue permutation preserving multiplicities of variants at each site (100,000 draws) and a TF bootstrap for observed/expected ratios (2,000 draws). The second compares PLP to BLB, VUS and POP with TF-stratified Mantel-Haenszel odds ratios; 95% percentile CIs resample TF clusters 20,000 times. `disjoint_sites` removes residue positions shared between clinical classes. Holm corrections cover three control comparisons separately per unit; informative TF counts may differ from 42 and 123. TF independence and lack of ascertainment correction remain limitations.

Dependencies: NumPy, pandas, SciPy, statsmodels, matplotlib. The notebook contains the analysis code and starts from prepared reference/variant/interface tables. Configure paths and run all cells. No ClinVar or structure downloads occur.

## Input layout for each cohort

`inputs/reference_sequences.json`: gene -> full-length amino-acid sequence.

`tables/variants_verified.tsv.gz`: `gene`, `group` (PLP/BLB/VUS/POP), `variant_id`, `position` (1-based), `ref`. Unique gene/group/variant IDs are required.

Interface CSV: `TF_Pair` (TF1_TF2), `TF{1,2}_seed{1..5}`, `TF{1,2}_Intersection`; comma-separated residues such as `A123`.

Use separate input roots for common and specific. Outputs and input hashes are written separately by cohort; reruns overwrite files in the configured output root.

In [ ]:
from pathlib import Path
COHORTS = {
    'specific': {'input': Path('/path/to/specific_cohort'),
                 'interfaces': Path('/path/to/interfaces/spe_1888_sites.csv'),
                 'network_tfs': 392, 'residues': 220080, 'all_interface_residues': 20766,
                 'plp_tfs': 123, 'plp_variants': 2073, 'plp_hits': 371},
    'common': {'input': Path('/path/to/common_cohort'),
               'interfaces': Path('/path/to/interfaces/com_339_sites.csv'),
               'network_tfs': 116, 'residues': 61627, 'all_interface_residues': 4600,
               'plp_tfs': 42, 'plp_variants': 841, 'plp_hits': 120},
}
OUTPUT_ROOT = Path('results/tf_interface_plp')
def require_inputs(paths):
    missing = [str(p) for p in paths if not p.exists()]
    if missing:
        raise FileNotFoundError("Configure the input paths first:\n" + "\n".join(missing))

def prepare_output(path, input_roots):
    path = path.resolve()
    for root in input_roots:
        root = root.resolve()
        if path == root or root in path.parents or path in root.parents:
            raise ValueError("Use an output directory separate from all input directories")
    path.mkdir(parents=True, exist_ok=True)

for cfg in COHORTS.values():
    require_inputs([cfg['input']/'inputs/reference_sequences.json',
                    cfg['input']/'tables/variants_verified.tsv.gz', cfg['interfaces']])
prepare_output(OUTPUT_ROOT, [c['input'] for c in COHORTS.values()] +
               [c['interfaces'].parent for c in COHORTS.values()])

## Interface enrichment against full-length sequence

The common and specific source implementations share the same algorithm. The optional legacy108 scope from the common script is omitted to keep these two P/LP cohorts explicit.

In [ ]:
def analyze_interface_enrichment(cfg, output):
    import os
    os.environ['OPENBLAS_NUM_THREADS']='1';os.environ.setdefault('MPLCONFIGDIR', str(output / 'matplotlib_cache'))
    from pathlib import Path
    from collections import defaultdict
    import json
    import numpy as np,pandas as pd
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    O=cfg['input'];D=output/'original_5of5_PLP';D.mkdir(parents=True,exist_ok=True)
    refs=json.load(open(O/'inputs/reference_sequences.json'));assert len(refs)==cfg['network_tfs'];pairs=pd.read_csv(cfg['interfaces']).fillna('');vv=pd.read_csv(O/'tables/variants_verified.tsv.gz',sep='\t');v=vv[vv.group=='PLP'].copy();ints=defaultdict(set)
    for r in pairs.to_dict('records'):
     for side,g in enumerate(r['TF_Pair'].split('_'),1):
      def ss(s):return {x.strip() for x in s.split(',') if x.strip()}
      inter=set.intersection(*[ss(r[f'TF{side}_seed{k}']) for k in range(1,6)])
      assert inter==ss(r[f'TF{side}_Intersection'])
      ints[g].update(inter)
    for g,ss in ints.items():
     for s in ss:assert 1<=int(s[1:])<=len(refs[g]) and refs[g][int(s[1:])-1]==s[0],(g,s)
    v['hit']=[int(f'{a}{int(p)}' in ints[g]) for g,p,a in zip(v.gene,v.position,v.ref)]
    v.to_csv(D/'PLP_variants.tsv',sep='\t',index=False)
    rows=[]
    for g,seq in refs.items():
     x=v[v.gene==g];s=x.drop_duplicates('position');rows.append({'gene':g,'length':len(seq),'interface_residues':len(ints[g]),'PLP_variants':len(x),'PLP_interface_variants':int(x.hit.sum()),'PLP_sites':len(s),'PLP_interface_sites':int(s.hit.sum()),'q':len(ints[g])/len(seq)})
    a=pd.DataFrame(rows).set_index('gene');a.to_csv(D/'per_TF.tsv',sep='\t');rng=np.random.default_rng(20260911);NBOOT=100000;result=[];per=[]
    scopes={'same_PLP_TFs':set(v.gene),'all_network_TFs':set(refs)}
    for scope,gs in scopes.items():
     bg=a.loc[sorted(gs)];case=bg[bg.PLP_variants>0];simv=np.zeros(NBOOT,dtype=int);sims=np.zeros(NBOOT,dtype=int)
     for g,z in case.iterrows():
      mult=v[v.gene==g].groupby('position').size().value_counts().sort_index();remaining=int(z.length);success=np.full(NBOOT,int(z.interface_residues),dtype=int)
      for weight,k in mult.items():
       draw=rng.hypergeometric(success,remaining-success,int(k));simv+=int(weight)*draw;sims+=draw;remaining-=int(k);success-=draw
     for unit,nn,oo,sim in [('variants','PLP_variants','PLP_interface_variants',simv),('sites','PLP_sites','PLP_interface_sites',sims)]:
      total=int(case[nn].sum());obs=int(case[oo].sum());M=int(bg.length.sum());I=int(bg.interface_residues.sum());q=I/M;E=float((case[nn]*case.q).sum());p=(1+sum(sim>=obs))/(NBOOT+1)
      idx=rng.integers(len(case),size=(2000,len(case)));ob=case[oo].values[idx].sum(1);ex=(case[nn]*case.q).values[idx].sum(1);valid=ex>0;lo,hi=np.quantile(ob[valid]/ex[valid],[.025,.975])
      result.append({'background':scope,'unit':unit,'background_TFs':len(bg),'PLP_TFs':len(case),'all_residues':M,'interface_residues':I,'interface_fraction':q,'PLP_total':total,'PLP_interface':obs,'PLP_fraction':obs/total,'pooled_OE':obs/total/q,'pooled_expected':total*q,'TF_stratified_expected':E,'TF_stratified_OE':obs/E,'TF_bootstrap_CI_low':lo,'TF_bootstrap_CI_high':hi,'permutation_P_upper':p,'MC_SE':np.sqrt(p*(1-p)/NBOOT),'null_mean':sim.mean(),'null_low':np.quantile(sim,.025),'null_high':np.quantile(sim,.975),'permutations':NBOOT})
     np.savez_compressed(D/(scope+'_null.npz'),variant_hits=simv,site_hits=sims)
    d=pd.DataFrame(result);d.to_csv(D/'enrichment_summary.tsv',sep='\t',index=False);print(d.to_string(index=False),flush=True)
    x=d[(d.background=='same_PLP_TFs')&(d.unit=='variants')].iloc[0];y=d[(d.background=='same_PLP_TFs')&(d.unit=='sites')].iloc[0];allr=d[(d.background=='all_network_TFs')&(d.unit=='variants')].iloc[0]
    plt.rcParams.update({'font.size':8,'pdf.fonttype':42});fig,ax=plt.subplots(figsize=(5,3));values=[x.interface_fraction,x.PLP_fraction,y.PLP_fraction];ax.bar(range(3),np.array(values)*100,color=['grey','steelblue','teal']);ax.set_xticks(range(3),['Interface / all residues','Interface P/LP variants','Interface P/LP sites']);ax.set_ylabel('Percentage (%)');ax.set_ylim(0,max(values)*100+4)
    for i,f in enumerate(values):ax.text(i,100*f+.4,f'{f:.2%}',ha='center')
    fig.tight_layout();fig.savefig(D/'original_5of5_PLP.pdf');fig.savefig(D/'original_5of5_PLP.png',dpi=180)

## TF-stratified pathogenicity comparisons

Input SHA-256 hashes are saved in `config.json`. Notebook source provenance is recorded below instead of using a script-only `__file__` reference.

In [ ]:
def analyze_pathogenicity(cfg, output):
    import os
    os.environ['OPENBLAS_NUM_THREADS']='1'
    os.environ.setdefault('MPLCONFIGDIR', str(output / 'matplotlib_cache'))
    from pathlib import Path
    from collections import defaultdict
    import json, hashlib, warnings
    import numpy as np
    import pandas as pd
    from scipy.stats import fisher_exact, t, chi2
    from statsmodels.stats.contingency_tables import StratifiedTable
    from statsmodels.stats.multitest import multipletests
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    O=cfg['input'];D=output/'original_5of5_pathogenicity';D.mkdir(parents=True,exist_ok=True)
    source=cfg['interfaces']
    refs=json.load(open(O/'inputs/reference_sequences.json'))
    v=pd.read_csv(O/'tables/variants_verified.tsv.gz',sep='\t')
    ints=defaultdict(set)
    def residues(s):return {x.strip() for x in str(s).split(',') if x.strip()}
    pairs=pd.read_csv(source).fillna('')
    for r in pairs.to_dict('records'):
     for side,g in enumerate(r['TF_Pair'].split('_'),1):
      ss=set.intersection(*[residues(r[f'TF{side}_seed{k}']) for k in range(1,6)])
      assert ss==residues(r[f'TF{side}_Intersection'])
      ints[g].update(ss)
    for g,ss in ints.items():
     for s in ss:assert refs[g][int(s[1:])-1]==s[0]
    assert not v.duplicated(['gene','group','variant_id']).any()
    assert all(1<=int(p)<=len(refs[g]) and refs[g][int(p)-1]==a for g,p,a in zip(v.gene,v.position,v.ref))
    v['hit']=[int(f'{a}{int(p)}' in ints[g]) for g,p,a in zip(v.gene,v.position,v.ref)]
    assert sum(map(len,ints.values()))==cfg['all_interface_residues']
    assert len(v[v.group=='PLP'])==cfg['plp_variants']
    assert v.loc[v.group=='PLP','gene'].nunique()==cfg['plp_tfs']
    assert v.loc[v.group=='PLP','hit'].sum()==cfg['plp_hits']
    v.to_csv(D/'variants_annotated.tsv.gz',sep='\t',index=False)
    bg=pd.DataFrame([dict(gene=g,length=len(s),interface_residues=len(ints[g])) for g,s in refs.items()])
    bg.to_csv(D/'full_length_background.tsv',sep='\t',index=False)
    summary=v.groupby('group').agg(variants=('hit','size'),interface_variants=('hit','sum'),TFs=('gene','nunique'))
    summary['interface_fraction']=summary.interface_variants/summary.variants
    summary.to_csv(D/'class_inventory.tsv',sep='\t')
    rng=np.random.default_rng(20260912);NBOOT=20000
    results=[];alltables=[];loos=[];sharedrows=[]
    # Each row: a=PLP interface,b=PLP noninterface,c=control interface,d=control noninterface.
    for ctrl in ['BLB','VUS','POP']:
     x=v[v.group.isin(['PLP',ctrl])].copy()
     assert not x.duplicated(['gene','variant_id']).any()
     shared=set(map(tuple,x[x.group=='PLP'][['gene','position']].values)) & set(map(tuple,x[x.group==ctrl][['gene','position']].values))
     for g,p in sorted(shared):sharedrows.append(dict(control=ctrl,gene=g,position=p,interface=int(f'{refs[g][int(p)-1]}{int(p)}' in ints[g])))
     for unit in ['variants','disjoint_sites']:
      y=x.copy()
      if unit=='disjoint_sites':
       y=y[[(g,p) not in shared for g,p in zip(y.gene,y.position)]].drop_duplicates(['gene','position','group'])
      genes=sorted(set(y[y.group=='PLP'].gene)&set(y[y.group==ctrl].gene))
      y=y[y.gene.isin(genes)]
      rr=[]
      for g in genes:
       z=y[y.gene==g];p=z[z.group=='PLP'];c=z[z.group==ctrl]
       a=int(p.hit.sum());b=len(p)-a;cc=int(c.hit.sum());d=len(c)-cc
       rr.append(dict(gene=g,a=a,b=b,c=cc,d=d,informative=(a+cc>0 and b+d>0)))
      tab=pd.DataFrame(rr);tab['control']=ctrl;tab['unit']=unit;alltables.append(tab)
      arr=tab[['a','b','c','d']].values.astype(float);total=arr.sum(0)
      inf=tab[tab.informative].copy();A=inf[['a','b','c','d']].values.astype(float)
      a,b,c,d=A.T;n=A.sum(1);U=a*d/n;V=b*c/n
      mh=U.sum()/V.sum();st=StratifiedTable(np.array([[[aa,bb],[cc,dd]] for aa,bb,cc,dd in A]).transpose(1,2,0),shift_zeros=False)
      assert np.isclose(mh,st.oddsratio_pooled)
      model_ci=st.oddsratio_pooled_confint()
      # Resample TF clusters with all observations retained, including uninformative TFs.
      aa,bb,cc,dd=arr.T;nn=arr.sum(1);ub=aa*dd/nn;vb=bb*cc/nn
      idx=rng.integers(len(tab),size=(NBOOT,len(tab)));num=ub[idx].sum(1);den=vb[idx].sum(1)
      valid=(den>0);boot=num[valid]/den[valid];lo,hi=np.quantile(boot,[.025,.975])
      # Cluster delta-method t inference for log ratio of MH estimating sums; independent TF clusters.
      G=len(tab);se=np.sqrt(G/(G-1)*np.sum((ub/ub.sum()-vb/vb.sum())**2))
      cp=2*t.sf(abs(np.log(mh)/se),df=G-1)
      pooled_or,pooled_p=fisher_exact(total.reshape(2,2),alternative='two-sided')
      for k,g in enumerate(tab.gene):
       un=ub.sum()-ub[k];vn=vb.sum()-vb[k]
       loos.append(dict(control=ctrl,unit=unit,omitted_gene=g,MH_OR=un/vn if vn else np.inf))
      with warnings.catch_warnings():
       warnings.simplefilter('ignore');bd=st.test_equal_odds(adjust=True).pvalue
      results.append(dict(control=ctrl,unit=unit,shared_sites_removed=len(shared) if unit=='disjoint_sites' else 0,TFs=G,informative_TFs=len(inf),PLP_interface=int(total[0]),PLP_noninterface=int(total[1]),control_interface=int(total[2]),control_noninterface=int(total[3]),PLP_interface_fraction=total[0]/sum(total[:2]),control_interface_fraction=total[2]/sum(total[2:]),pooled_OR=pooled_or,pooled_Fisher_P=pooled_p,MH_OR=mh,TF_bootstrap_CI_low=lo,TF_bootstrap_CI_high=hi,bootstrap_valid=int(valid.sum()),bootstrap_total=NBOOT,cluster_logOR_SE=se,cluster_t_P=cp,CMH_P=chi2.sf(st.test_null_odds(correction=False).statistic,1),CMH_CI_low=model_ci[0],CMH_CI_high=model_ci[1],Breslow_Day_P=bd))
    r=pd.DataFrame(results)
    for unit in r.unit.unique():
     mask=r.unit==unit
     for pcol in ['cluster_t_P','CMH_P']:
      r.loc[mask,pcol+'_Holm']=multipletests(r.loc[mask,pcol],method='holm')[1]
    r.to_csv(D/'comparison_summary.tsv',sep='\t',index=False)
    pd.concat(alltables).to_csv(D/'per_TF_contingency.tsv',sep='\t',index=False)
    pd.DataFrame(loos).to_csv(D/'leave_one_TF_out.tsv',sep='\t',index=False)
    pd.DataFrame(sharedrows).to_csv(D/'shared_PLP_control_sites.tsv',sep='\t',index=False)
    # Source manifests and exact analysis configuration.
    paths=[source,O/'inputs/reference_sequences.json',O/'tables/variants_verified.tsv.gz']
    manifest={str(p):hashlib.sha256(p.read_bytes()).hexdigest() for p in paths}
    json.dump(dict(inputs_sha256=manifest,interface='cross-chain all-atom <=5A; 5/5 per-pair intersection; union across partners',confidence_filter=False,SASA_filter=False,background=f"full reference sequence; {cfg['network_tfs']} TFs; {cfg['residues']} residues",clinical_scope='verified simple missense, all origins, no review filter, standard PLP/BLB/VUS; no structural filter',population='cached gnomAD r4 exome PASS, AC>0, AN>=80% gene max; known ClinVar PLP genomic alleles excluded upstream',seed=20260912,bootstrap=NBOOT,tests='two-sided; Holm across 3 comparisons separately per unit; TF cluster bootstrap percentile CI; independent TF cluster assumption'),open(D/'config.json','w'),indent=2)
    plt.rcParams.update({'font.family':'DejaVu Sans','font.size':7,'pdf.fonttype':42})
    fig,ax=plt.subplots(figsize=(5.8,3.4))
    for i,z in r.iterrows():
     color='#276FA3' if z.unit=='variants' else '#B56532'
     ax.errorbar(z.MH_OR,5-i,xerr=[[z.MH_OR-z.TF_bootstrap_CI_low],[z.TF_bootstrap_CI_high-z.MH_OR]],fmt='o',color=color,capsize=3)
    ax.set_yticks(range(6),[f"P/LP vs {z.control} | {z.unit.replace('_',' ')}" for _,z in r.iloc[::-1].iterrows()])
    ax.axvline(1,color='grey',ls='--',lw=.8);ax.set_xscale('log');ax.set_xlabel('TF-stratified MH odds ratio (95% TF bootstrap CI)')
    fig.tight_layout();fig.savefig(D/'pathogenicity_OR.pdf');fig.savefig(D/'pathogenicity_OR.png',dpi=180)
    print(summary.to_string());print(r.to_string(index=False));print('COMPLETE',D,flush=True)

## Execute each cohort separately

In [ ]:
for cohort, cfg in COHORTS.items():
    output = OUTPUT_ROOT / cohort
    output.mkdir(parents=True, exist_ok=True)
    print(cohort, 'P/LP TFs:', cfg['plp_tfs'])
    analyze_interface_enrichment(cfg, output)
    analyze_pathogenicity(cfg, output)

## Source provenance

Adapted from the following analysis scripts. SHA-256 values identify the original local source snapshots, before path portability changes. Source data are not bundled.

| Source script | SHA-256 |
| --- | --- |
| `13_original_interface_PLP.py` | `8628d2af5c7cc80cfada620839fa38adc32bae69e7f1e4f46429e1f9553a25a8` |
| `14_original_pathogenicity.py` | `7aba6e5be3dea5133978025055a43f887004f719cbc9a806e1637fff6b75e4f4` |
| `13_original_interface_PLP.py` | `97c9ea0ccd7180a50eee230e1ea06f787ee93896c1ff1d91084cd547047e1583` |
| `14_original_pathogenicity.py` | `3e00b414ffe8cdfeb45c8277835519204153b5766b85af5b5494b7748b584023` |